# 🌻 Flower Classification with MobileNetV2
**Dataset:** [Kaggle · Flowers Recognition](https://www.kaggle.com/datasets/alxmamaev/flowers-recognition) (~4.3k photos, 5 classes: daisy · dandelion · rose · sunflower · tulip)

**What this notebook does**
1. Downloads the dataset with `kagglehub` (no API key needed for public datasets)
2. Builds a `tf.data` pipeline with an **on-model data-augmentation** stage
3. **Transfer learning**: frozen MobileNetV2 (ImageNet) + a small classification head
4. **Fine-tuning**: unfreezes the top of the backbone at a low learning rate
5. **Evaluation**: accuracy/loss curves, classification report, confusion matrix, misclassified gallery
6. **Inference** on your own uploaded photo + optional **TFLite export**

> ⚡ Runtime → Change runtime type → **GPU (T4)** before running. Full run ≈ 10–15 min on a T4.


## 1 · Setup

In [ ]:
!pip -q install kagglehub

In [ ]:
import os, pathlib, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU') or "none — enable it in Runtime > Change runtime type")

## 2 · Download the dataset
`kagglehub` fetches public Kaggle datasets without credentials and caches them locally.

In [ ]:
import kagglehub

root = kagglehub.dataset_download("alxmamaev/flowers-recognition")
print("Dataset downloaded to:", root)

In [ ]:
CLASSES = ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']

def find_data_dir(root):
    """Locate the directory that directly contains the 5 class folders
    (the archive sometimes nests flowers/flowers/...)."""
    for dirpath, dirnames, _ in os.walk(root):
        if all(c in dirnames for c in CLASSES):
            return pathlib.Path(dirpath)
    raise FileNotFoundError("Could not find the 5 class folders under " + str(root))

data_dir = find_data_dir(root)
print("Class folders live in:", data_dir)

for c in CLASSES:
    n = len(list((data_dir / c).glob('*.jpg')))
    print(f"  {c:<10} {n:>5} images " + "█" * (n // 40))

### A first look

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(14, 8))
for col, c in enumerate(CLASSES):
    picks = random.sample(list((data_dir / c).glob('*.jpg')), 3)
    for row, p in enumerate(picks):
        ax = axes[row, col]
        ax.imshow(plt.imread(p)); ax.axis('off')
        if row == 0: ax.set_title(c, fontsize=13)
plt.suptitle("Random samples per class", y=1.02)
plt.tight_layout(); plt.show()

## 3 · tf.data pipeline
80/20 train/validation split. The validation set is loaded with `shuffle=False` so predictions stay aligned with labels for the confusion matrix later.

In [ ]:
IMG_SIZE = (224, 224)
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset='training', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode='int')

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode='int', shuffle=False)

class_names = train_ds.class_names
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## 4 · Data augmentation
Augmentation lives **inside the model** as Keras layers — it runs on the GPU and is automatically disabled at inference time.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.12),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomTranslation(0.08, 0.08),
    tf.keras.layers.RandomContrast(0.15),
], name='augmentation')

# visualize: one image, eight augmented views
for images, _ in train_ds.take(1):
    plt.figure(figsize=(14, 4))
    ax = plt.subplot(1, 9, 1)
    plt.imshow(images[0].numpy().astype('uint8')); plt.title('original'); plt.axis('off')
    for i in range(8):
        aug = data_augmentation(tf.expand_dims(images[0], 0), training=True)
        ax = plt.subplot(1, 9, i + 2)
        plt.imshow(tf.clip_by_value(aug[0], 0, 255).numpy().astype('uint8'))
        plt.title(f'aug {i+1}', fontsize=9); plt.axis('off')
    plt.tight_layout(); plt.show()

## 5 · Transfer learning — MobileNetV2
The ImageNet-pretrained backbone is **frozen**; we only train a small head on top.
`preprocess_input` (scales pixels to −1…1, what MobileNetV2 expects) is part of the graph, so the saved model accepts raw 0–255 images.

In [ ]:
base = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)          # training=False also freezes BatchNorm statistics
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name='flower_mobilenetv2')
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary(show_trainable=True)

## 6 · Train the head

In [ ]:
EPOCHS_HEAD = 10

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.4,
                                         patience=2, min_lr=1e-6, verbose=1),
]

history = model.fit(train_ds, validation_data=val_ds,
                    epochs=EPOCHS_HEAD, callbacks=callbacks)

## 7 · Fine-tune the top of the backbone
Unfreeze everything from layer 100 up (~55 of 154 layers) and train at a **much lower** learning rate so we nudge, not destroy, the pretrained features.

In [ ]:
base.trainable = True
FINE_TUNE_AT = 100
for layer in base.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

EPOCHS_FT = 8
total_epochs = len(history.epoch) + EPOCHS_FT

history_ft = model.fit(train_ds, validation_data=val_ds,
                       epochs=total_epochs, initial_epoch=len(history.epoch),
                       callbacks=callbacks)

## 8 · Evaluation — learning curves

In [ ]:
acc  = history.history['accuracy'] + history_ft.history['accuracy']
vacc = history.history['val_accuracy'] + history_ft.history['val_accuracy']
loss = history.history['loss'] + history_ft.history['loss']
vloss = history.history['val_loss'] + history_ft.history['val_loss']
ft_start = len(history.history['accuracy'])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
a1.plot(acc, label='train'); a1.plot(vacc, label='validation')
a1.axvline(ft_start - 0.5, ls='--', c='gray'); a1.text(ft_start, min(acc), ' fine-tune →', fontsize=9)
a1.set_title('Accuracy'); a1.set_xlabel('epoch'); a1.legend(); a1.grid(alpha=0.3)
a2.plot(loss, label='train'); a2.plot(vloss, label='validation')
a2.axvline(ft_start - 0.5, ls='--', c='gray')
a2.set_title('Loss'); a2.set_xlabel('epoch'); a2.legend(); a2.grid(alpha=0.3)
plt.show()

val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f"Final validation accuracy: {val_acc:.2%}   (loss {val_loss:.4f})")

### Classification report & confusion matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = np.concatenate([y.numpy() for _, y in val_ds])
y_prob = model.predict(val_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap='Greens')
ax.set_xticks(range(len(class_names)), class_names, rotation=30, ha='right')
ax.set_yticks(range(len(class_names)), class_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title('Confusion matrix (validation)')
fig.colorbar(im, shrink=0.8); plt.tight_layout(); plt.show()

### Where does it fail? — misclassified gallery

In [ ]:
val_images = np.concatenate([x.numpy() for x, _ in val_ds]).astype('uint8')
wrong = np.where(y_pred != y_true)[0]
print(f"{len(wrong)} / {len(y_true)} validation images misclassified")

show = wrong[np.argsort(-y_prob[wrong].max(axis=1))][:12]   # most confident mistakes first
if len(show):
    fig, axes = plt.subplots(2, 6, figsize=(16, 6))
    for ax, idx in zip(axes.flat, show):
        ax.imshow(val_images[idx]); ax.axis('off')
        ax.set_title(f"true: {class_names[y_true[idx]]}\npred: {class_names[y_pred[idx]]} "
                     f"({y_prob[idx].max():.0%})", fontsize=9)
    for ax in axes.flat[len(show):]: ax.axis('off')
    plt.suptitle('Most confident mistakes', y=1.03); plt.tight_layout(); plt.show()

## 9 · Try your own photo
Run the cell, pick a flower photo from your device, get the top-3 guesses.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        img = tf.keras.utils.load_img(name, target_size=IMG_SIZE)
        arr = tf.keras.utils.img_to_array(img)[None, ...]
        probs = model.predict(arr, verbose=0)[0]
        top3 = probs.argsort()[::-1][:3]
        plt.figure(figsize=(3.2, 3.2)); plt.imshow(img); plt.axis('off')
        plt.title("\n".join(f"{class_names[i]}: {probs[i]:.1%}" for i in top3), fontsize=10)
        plt.show()
except ImportError:
    print("Not running in Colab — point `name` at a local file instead.")

## 10 · Save & export
Saves the full Keras model and converts to **TFLite** (MobileNetV2's natural habitat — phones and edge devices).

In [ ]:
model.save('flower_mobilenetv2.keras')
print("Saved flower_mobilenetv2.keras "
      f"({os.path.getsize('flower_mobilenetv2.keras') / 1e6:.1f} MB)")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open('flower_mobilenetv2.tflite', 'wb') as f:
    f.write(tflite_model)
print(f"Saved flower_mobilenetv2.tflite ({len(tflite_model) / 1e6:.1f} MB, quantized)")

# In Colab: download via the files panel on the left, or:
# from google.colab import files; files.download('flower_mobilenetv2.tflite')

---
### Ideas to push further
- Swap the backbone: `EfficientNetV2B0` usually adds a couple of points
- Class weights or oversampling (dandelion has the most images, rose the fewest)
- `tf.keras.layers.RandomBrightness` for tougher lighting robustness
- Grad-CAM to see *where* the model looks before it says "tulip"

*Notebook crafted at sloppy.live — vibecoded live with chat 🌻*